In [1]:
import gc
import json
import os
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch

from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

PROJECT_ROOT = Path(r"D:\\dev\\projects\\fourlang_translation")
TRAIN_FILE = PROJECT_ROOT / "data/clean/en_uz/hplt/directional/train.jsonl"
VALID_FILE = PROJECT_ROOT / "data/clean/en_uz/hplt/directional/validation.jsonl"
TEST_FILE = PROJECT_ROOT / "data/clean/en_uz/hplt/directional/test.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "models/lora/m2m100_en_uz_hplt_smoke"
RESULT_DIR = PROJECT_ROOT / "results/m2m100_en_uz_hplt_smoke"

BASE_MODEL = "facebook/m2m100_418M"
SEED = 42
MAX_TRAIN_SAMPLES = 10_000
MAX_VALID_SAMPLES = 1_000
MAX_TEST_SAMPLES = 400
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
MAX_STEPS = 500

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_num_threads(4)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Project:", PROJECT_ROOT)
print("Output :", OUTPUT_DIR)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: D:\dev\projects\fourlang_translation
Output : D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_hplt_smoke


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("BF16:", USE_BF16, "FP16:", USE_FP16)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"Dedicated VRAM: {props.total_memory / 1024**3:.2f} GB")

assert torch.cuda.is_available(), "未检测到 CUDA GPU，请检查 PyTorch/CUDA 环境"

PyTorch: 2.13.0+cu132
CUDA available: True
Device: cuda
BF16: True FP16: False
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Dedicated VRAM: 7.96 GB


In [3]:
for file in (TRAIN_FILE, VALID_FILE, TEST_FILE):
    assert file.exists(), f"找不到数据文件: {file}"

train_dataset = load_dataset("json", data_files=str(TRAIN_FILE), split="train")
valid_dataset = load_dataset("json", data_files=str(VALID_FILE), split="train")
test_dataset = load_dataset("json", data_files=str(TEST_FILE), split="train")

train_dataset = train_dataset.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
valid_dataset = valid_dataset.shuffle(seed=SEED).select(range(min(MAX_VALID_SAMPLES, len(valid_dataset))))
test_dataset = test_dataset.shuffle(seed=SEED).select(range(min(MAX_TEST_SAMPLES, len(test_dataset))))

REQUIRED_COLUMNS = {"src_lang", "tgt_lang", "src_text", "tgt_text"}
for name, dataset in (("train", train_dataset), ("valid", valid_dataset), ("test", test_dataset)):
    missing = REQUIRED_COLUMNS.difference(dataset.column_names)
    assert not missing, f"{name} 缺少字段: {sorted(missing)}"
    languages = set(dataset.unique("src_lang")) | set(dataset.unique("tgt_lang"))
    assert languages <= {"en", "uz"}, f"发现未知语言: {languages}"
    print(name, len(dataset), Counter(f"{x['src_lang']}-{x['tgt_lang']}" for x in dataset))

train 10000 Counter({'en-uz': 5005, 'uz-en': 4995})
valid 1000 Counter({'en-uz': 503, 'uz-en': 497})
test 400 Counter({'en-uz': 202, 'uz-en': 198})


In [4]:
CYRILLIC_RE = re.compile(r"[\u0400-\u04FF]")
LATIN_RE = re.compile(r"[A-Za-z]")

def dominant_script(text):
    cyrillic = len(CYRILLIC_RE.findall(text))
    latin = len(LATIN_RE.findall(text))
    if not cyrillic and not latin:
        return "other"
    return "cyrillic" if cyrillic > latin else "latin"

def audit_dataset(dataset):
    pairs = []
    empty = identical = 0
    uz_scripts = Counter()
    for row in dataset:
        source = row["src_text"].strip()
        target = row["tgt_text"].strip()
        empty += int(not source or not target)
        identical += int(bool(source) and source.casefold() == target.casefold())
        pairs.append((row["src_lang"], row["tgt_lang"], source, target))
        if row["src_lang"] == "uz":
            uz_scripts[dominant_script(source)] += 1
        if row["tgt_lang"] == "uz":
            uz_scripts[dominant_script(target)] += 1
    return {
        "samples": len(dataset),
        "empty": empty,
        "identical": identical,
        "exact_duplicates": len(pairs) - len(set(pairs)),
        "uz_scripts": dict(uz_scripts),
    }

audit = {
    "train": audit_dataset(train_dataset),
    "validation": audit_dataset(valid_dataset),
    "test": audit_dataset(test_dataset),
}
print(json.dumps(audit, ensure_ascii=False, indent=2))
(RESULT_DIR / "data_audit.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame(train_dataset.select(range(min(10, len(train_dataset))))).head(10)

{
  "train": {
    "samples": 10000,
    "empty": 0,
    "identical": 0,
    "exact_duplicates": 0,
    "uz_scripts": {
      "latin": 10000
    }
  },
  "validation": {
    "samples": 1000,
    "empty": 0,
    "identical": 0,
    "exact_duplicates": 0,
    "uz_scripts": {
      "latin": 1000
    }
  },
  "test": {
    "samples": 400,
    "empty": 0,
    "identical": 0,
    "exact_duplicates": 0,
    "uz_scripts": {
      "latin": 400
    }
  }
}


,pair_id,src_lang,tgt_lang,src_text,tgt_text,direction,split
0,hplt_00005971,uz,en,"Senga Robbingdan vahiy qilingan narsaga, u ko'...",Is there anyone who knows that what is reveale...,uz-en,train
1,hplt_00003484,uz,en,- Besh qurbonlik;,- Five sacrals;,uz-en,train
2,hplt_00009448,uz,en,Unga o'xshash hech narsa yo'qdir.,There is nothing like Him.,uz-en,train
3,hplt_00008974,uz,en,[8] Big yoki kichik.,[8] Big or small.,uz-en,train
4,hplt_00006288,uz,en,"Va yana, so'zma-so'z o'ylamang, o'ylang, bilas...","And again, don't think literally, think, you k...",uz-en,train
5,hplt_00000658,en,uz,He left gaps for undiscovered elements but nev...,U ochilmagan elementlar uchun bo'shliqlar qold...,en-uz,train
6,hplt_00006858,uz,en,Dragon Ball Fighter Z: yakuniy DATAMINE & komp...,Dragon Ball Fighter Z: Final Datamine & PCs 1.,uz-en,train
7,hplt_00000329,uz,en,"Qachon kimdir OCPD ega, kichik narsalar juda k...","When someone has OCPD, take small things too m...",uz-en,train
8,hplt_00008145,en,uz,"Thank you, However I am having problems with y...","Rahmat, lekin men sizning RSS bilan bog'liq mu...",en-uz,train
9,hplt_00008398,uz,en,"iqlim: Yumshoq, quruq, ba'zan shamol.","climate: Soft, dry, sometimes wind.",uz-en,train


In [5]:
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    low_cpu_mem_usage=True,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.config.use_cache = False
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
assert trainable / total < 0.02, "可训练参数比例异常，可能没有正确冻结基础模型"

trainable params: 2,359,296 || all params: 486,264,832 || trainable%: 0.4852


In [6]:
def preprocess(example):
    tokenizer.src_lang = example["src_lang"]
    model_inputs = tokenizer(
        example["src_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    tokenizer.src_lang = example["tgt_lang"]
    labels = tokenizer(
        example["tgt_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess, remove_columns=train_dataset.column_names, desc="Tokenize train"
)
tokenized_valid = valid_dataset.map(
    preprocess, remove_columns=valid_dataset.column_names, desc="Tokenize valid"
)

sample_index = 0
print("INPUT :", tokenizer.decode(tokenized_train[sample_index]["input_ids"], skip_special_tokens=False))
print("LABEL :", tokenizer.decode(tokenized_train[sample_index]["labels"], skip_special_tokens=False))

Tokenize valid: 100%|██████████| 1000/1000 [00:00<00:00, 4935.73 examples/s]

INPUT : __uz__ Senga Robbingdan vahiy qilingan narsaga, u ko'r bo'lib shaxs sifatida bir xil ekanini biladi hech kim bormi? [Ar Ra`ad: 19].</s>
LABEL : __en__ Is there anyone who knows that what is revealed to you from your Lord, it was the same as the person who is blind?</s>


In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    bf16=USE_BF16,
    fp16=USE_FP16,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    predict_with_generate=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    save_safetensors=True,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=collator,
)

C:\Users\WingYouther\AppData\Local\Temp\ipykernel_87976\3199607509.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
max_steps is given, it will override any value given in num_train_epochs


In [8]:
model = model.to(DEVICE)
model.train()
torch.cuda.reset_peak_memory_stats()
batch = collator([tokenized_train[0]])
batch = {key: value.to(DEVICE) for key, value in batch.items()}

with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    enabled=DEVICE.type == "cuda",
):
    outputs = model(**batch)
    loss = outputs.loss

assert torch.isfinite(loss), f"Loss 异常: {loss.item()}"
assert loss.requires_grad, "Loss 没有梯度，请重启 Kernel 后从第一个单元格重新运行"
trainer.accelerator.backward(loss)
model.zero_grad(set_to_none=True)
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"Preflight loss: {loss.item():.4f}")
print(f"Peak allocated VRAM: {peak_gb:.2f} GB")
assert peak_gb < 7.5, "显存峰值过高：请降低 MAX_SOURCE_LENGTH/MAX_TARGET_LENGTH"

del batch, outputs, loss
gc.collect()
torch.cuda.empty_cache()

Preflight loss: 3.4255
Peak allocated VRAM: 2.15 GB


In [9]:
@torch.inference_mode()
def translate(text, src_lang, tgt_lang, model_obj=model):
    model_obj.eval()
    model_obj.config.use_cache = True
    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=MAX_SOURCE_LENGTH
    ).to(DEVICE)
    generated = model_obj.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=MAX_TARGET_LENGTH,
        num_beams=1,
    )
    model_obj.config.use_cache = False
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

baseline_examples = [
    ("en", "uz", "I will go to the airport tomorrow morning."),
    ("uz", "en", "Men ertaga ertalab aeroportga boraman."),
]
for src_lang, tgt_lang, text in baseline_examples:
    print(src_lang, "->", tgt_lang, "|", text, "=>", translate(text, src_lang, tgt_lang))

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


en -> uz | I will go to the airport tomorrow morning. => Yarın sabah aerodinamikga qaytarga.
uz -> en | Men ertaga ertalab aeroportga boraman. => It is also the airport.


In [10]:
RESUME_FROM_CHECKPOINT = False
model.config.use_cache = False
train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
trainer.save_model(str(OUTPUT_DIR / "final_adapter"))
tokenizer.save_pretrained(OUTPUT_DIR / "final_adapter")
trainer.save_metrics("train", train_result.metrics)
print("Adapter saved to:", OUTPUT_DIR / "final_adapter")

Step,Training Loss,Validation Loss
100,4.433300,4.093126
200,4.070700,3.950643
300,4.042400,3.874126
400,4.276800,3.833540
500,3.980400,3.813825


Adapter saved to: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_hplt_smoke\final_adapter


In [11]:
validation_metrics = trainer.evaluate()
trainer.save_metrics("eval", validation_metrics)
validation_metrics

{'eval_loss': 3.8138246536254883,
 'eval_runtime': 33.8448,
 'eval_samples_per_second': 29.547,
 'eval_steps_per_second': 29.547,
 'epoch': 0.8}

In [12]:
model.eval()
model.config.use_cache = True
_ = translate("Hello.", "en", "uz")
predictions = []

with torch.inference_mode():
    for row in test_dataset:
        tokenizer.src_lang = row["src_lang"]
        inputs = tokenizer(
            row["src_text"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
        ).to(DEVICE)
        torch.cuda.synchronize()
        started = time.perf_counter()
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.get_lang_id(row["tgt_lang"]),
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=1,
        )
        torch.cuda.synchronize()
        predictions.append({
            "direction": f"{row['src_lang']}-{row['tgt_lang']}",
            "source": row["src_text"],
            "reference": row["tgt_text"],
            "prediction": tokenizer.batch_decode(generated, skip_special_tokens=True)[0],
            "latency_seconds": time.perf_counter() - started,
        })

prediction_df = pd.DataFrame(predictions)
metrics = {}
for direction, group in prediction_df.groupby("direction"):
    preds = group["prediction"].tolist()
    refs = group["reference"].tolist()
    metrics[direction] = {
        "samples": len(group),
        "bleu": sacrebleu.corpus_bleu(preds, [refs]).score,
        "chrf2": sacrebleu.corpus_chrf(preds, [refs], word_order=2).score,
        "latency_mean_seconds": group["latency_seconds"].mean(),
        "latency_p95_seconds": group["latency_seconds"].quantile(0.95),
    }

prediction_df.to_csv(RESULT_DIR / "predictions_after.csv", index=False, encoding="utf-8-sig")
(RESULT_DIR / "metrics_after.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
prediction_df.head(20)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\utils.py:1493: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


{
  "en-uz": {
    "samples": 202,
    "bleu": 1.9725760306547255,
    "chrf2": 12.786916960359171,
    "latency_mean_seconds": 0.8979684207974564,
    "latency_p95_seconds": 1.5713119200372603
  },
  "uz-en": {
    "samples": 198,
    "bleu": 8.265192510560716,
    "chrf2": 29.493473423479678,
    "latency_mean_seconds": 0.338309919192324,
    "latency_p95_seconds": 0.6443106450140468
  }
}


,direction,source,reference,prediction,latency_seconds
0,uz-en,Aristokratens Buffalo uyasi onlayn o'ynash mum...,Aristokratens Buffalo slot is available to pla...,Aristocratus Buffalo is a real or real player ...,0.319596
1,uz-en,U va do'stlar Atlantika Siti shahrida kazino o...,He and friends played casinos in Atlantic City...,You and the fans of the Atlantic City casino h...,0.691348
2,uz-en,"Kafedra o’qituvchilari AQSH, Moskva, Sankt-Pet...",Teachers of the department participated in int...,The department has been presented in the vario...,0.448595
3,uz-en,Ushbu uslub tobora ko'proq qo'llanilmoqda va s...,This technique is more and more used and has p...,What is the case that you have been presented ...,0.365713
4,en-uz,- Journalism.,- Shaxsiy sahifalar.,- Gazetiyizm.,0.097940
5,en-uz,Inquisitive minds want to know.,Noma'lum aqllar bilishni xohlashadi.,Inquisitiv menti bildi.,0.135986
6,en-uz,The dashboard provides access to GPS navigatio...,"Qurilma paneli GPS-navigatsiya, GSM, 3G / 4G, ...","Dashboard GPS navigation, GSM, 3G/4G, bluetoot...",0.497449
7,uz-en,FISAning shtab-kvartirasi 1922-yilda Shveysari...,"FISA established its headquarters in Lausanne,...",The establishment of the FISA was established ...,0.336669
8,en-uz,The Pet Place editors have asked me to give th...,Pet Place muharrirlari mendan ushbu savolga bi...,Pet Place editörlar bu o'zni o'zdan o'zdan o'z...,1.512328
9,uz-en,Agar siz ma'lum bir mavzu yoki hisobot turi uc...,If you have created a presentation folder for ...,If you want to create a new version or a new v...,0.395530


In [13]:
adapter_dir = OUTPUT_DIR / "final_adapter"
assert adapter_dir.exists(), f"Adapter 不存在: {adapter_dir}"

reload_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
reload_base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, low_cpu_mem_usage=True)
reload_model = PeftModel.from_pretrained(reload_base, adapter_dir).to(DEVICE).eval()
reload_model.config.use_cache = True

def translate_reloaded(text, src_lang, tgt_lang):
    reload_tokenizer.src_lang = src_lang
    inputs = reload_tokenizer(text, return_tensors="pt", truncation=True, max_length=96).to(DEVICE)
    with torch.inference_mode():
        generated = reload_model.generate(
            **inputs,
            forced_bos_token_id=reload_tokenizer.get_lang_id(tgt_lang),
            max_new_tokens=96,
            num_beams=1,
        )
    return reload_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

print(translate_reloaded("Men ertaga aeroportga boraman.", "uz", "en"))
print(translate_reloaded("I will go to the airport tomorrow.", "en", "uz"))

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


But I stay at the airport.
Yarın havayolanda ko'zim.
